# Stochastic Uncertainty Around the LSTM Forecast

A point forecast gives one expected future yield curve. A probabilistic forecast asks: what are plausible yield curves around that point forecast?

This notebook uses validation residuals to build a simple uncertainty model and then reads the saved Monte Carlo outputs.

## Key terms

- **Residual**: actual change minus predicted change.
- **Variance**: how spread out one maturity's errors are.
- **Covariance**: how two maturities' errors move together.
- **Covariance matrix**: a table containing all variances and covariances across maturities.
- **Multivariate normal distribution**: a probability model that samples several related variables at once.
- **Point forecast**: one best estimate.
- **Probabilistic forecast**: a range of plausible outcomes around the point estimate.

## Assumptions

This is intentionally simple. It assumes validation residuals are representative of near-future errors, residuals are centered at zero, errors are approximately multivariate normal, and the covariance matrix is stable for this one-step forecast.

In [ ]:
# ruff: noqa: E402, I001
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SUMMARY_PATH = PROJECT_ROOT / "results" / "uncertainty_summary.json"
SCENARIO_PATH = PROJECT_ROOT / "results" / "monte_carlo_yield_curves.csv"
RANGE_PATH = PROJECT_ROOT / "reports" / "tables" / "uncertainty_ranges.csv"
FAN_FIGURE = PROJECT_ROOT / "reports" / "figures" / "yield_curve_fan.png"
RANGE_FIGURE = (
    PROJECT_ROOT / "reports" / "figures" / "uncertainty_ranges_by_maturity.png"
)

## Load saved uncertainty outputs

The simulation generated approximately 1,000 future yield curves and saved confidence ranges by maturity.

In [ ]:
summary = json.loads(SUMMARY_PATH.read_text())
scenarios = pd.read_csv(SCENARIO_PATH)
ranges = pd.read_csv(RANGE_PATH)
{
    "latest_input_date": summary["latest_input_date"],
    "scenario_count": summary["scenario_count"],
    "residual_source": summary["residual_source"],
    "scenario_table_shape": scenarios.shape,
}

## Residual covariance matrix

This matrix is estimated from validation residuals only. Values are in squared basis-point units. Larger diagonal values mean that maturity has more forecast-error variance. Positive off-diagonal values mean errors tend to move together.

In [ ]:
maturities = list(summary["point_forecast_yields_percent"].keys())
covariance = pd.DataFrame(
    summary["residual_covariance_basis_points"],
    index=maturities,
    columns=maturities,
)
covariance

## Yield-curve fan

The fan chart shows the neural point forecast plus Monte Carlo uncertainty bands. Wider bands mean more uncertainty.

In [ ]:
display(Image(filename=FAN_FIGURE))

## Confidence ranges by maturity

The table shows the 5th, 50th, and 95th percentile simulated yield levels. The 90% width is shown in basis points.

In [ ]:
ranges

In [ ]:
display(Image(filename=RANGE_FIGURE))

## Example simulated curves

Each row is one simulated future yield curve. The first column is just the scenario number.

In [ ]:
scenarios.head(10)

## Interpretation

This is a lightweight uncertainty estimate, not a full market risk engine. Its value is that it turns one forecast into a range of plausible curves while preserving the observed cross-maturity error structure from validation residuals.